In [2]:
import pandas as pd
import yaml
from pathlib import Path

root_folder = Path(r"/Users/panyue/Desktop/final_data/1_crop_management_data")

# only keep files whose name contains this text
name_filter = "crop_registration_basic"

In [3]:
wanted_columns = [
    "ID_all",
    "year",
    "crop",
    "variety",
    "date_planting",
    "date_harvest"
]

output_dir = Path("/Users/panyue/PycharmProjects/wofost_example_test/input") / "agro"
output_dir.mkdir(parents=True, exist_ok=True)

excel_files = [
    f for f in root_folder.rglob("*.xlsx")
    if not f.name.startswith("~$")
    and name_filter.lower() in f.name.lower()
]

for file in excel_files:
    try:
        df = pd.read_excel(file)

        # clean column names
        df.columns = df.columns.astype(str).str.strip()

        # optional: standardize common variants
        df = df.rename(columns={
            "Yield": "yield",
            "YIELD": "yield",
            "yield ": "yield",
            " date_planting": "date_planting",
            " date_harvest": "date_harvest"
        })

        missing = [col for col in wanted_columns if col not in df.columns]
        if missing:
            print(f"Skipped {file.name}: missing columns {missing}")
            print("Columns found:", df.columns.tolist())
            continue

        df = df[wanted_columns].copy()

        # clean dates
        for col in ["date_planting", "date_harvest"]:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
            df[col] = df[col].dt.strftime("%Y-%m-%d")

        field_name = None
        if "ID_field" in df.columns:
            non_null_fields = df["ID_field"].dropna()
            if not non_null_fields.empty:
                field_name = str(non_null_fields.iloc[0])
        if not field_name:
            field_name = file.stem

        # Create WOFOST-compatible agro management structure
        agro_management = []
        
        for idx, row in df.iterrows():
            # Use planting date as the key for AgroManagement
            planting_date = row['date_planting']
            
            agro_entry = {
                planting_date: {
                    "CropCalendar": {
                        "crop_start_date": row['date_planting'],
                        "crop_start_type": "sowing",
                        "crop_end_date": row['date_harvest'],
                        "crop_end_type": "harvest",
                        "crop_name": row['crop'].lower(),
                        "variety_name": row['variety'],
                        "max_duration": 365
                    },
                    "StateEvents": None,
                    "TimedEvents": {}
                }
            }
            agro_management.append(agro_entry)
        
        # Create the final YAML structure
        output_data = {
            "AgroManagement": agro_management,
            "Version": "1.0"
        }

        output_path = output_dir / f"agro_{field_name}.yaml"
        with open(output_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(output_data, f, sort_keys=False, default_flow_style=False)
        print(f"Wrote {output_path}")

    except Exception as e:
        print(f"Skipped {file}: {e}")

Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_DR14.yaml
Skipped /Users/panyue/Desktop/final_data/1_crop_management_data/ZW12/crop_registration_basic_ZW12.xlsx: 'float' object has no attribute 'lower'
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_DR13.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_ZW15.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_ZW22.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_DR12.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_ZW14.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_DR15.yaml
Skipped /Users/panyue/Desktop/final_data/1_crop_management_data/ZW13/crop_registration_basic_ZW13.xlsx: 'float' ob

/Users/panyue/Library/Python/3.14/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/Users/panyue/Library/Python/3.14/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [15]:
import yaml
from pathlib import Path

# Directory containing the YAML files
input_dir = Path("/Users/panyue/PycharmProjects/wofost_example_test/input/agro")

# List to store all variety names for crop_name = potato
potato_varieties = []

# Iterate through all YAML files in the directory
for yaml_file in input_dir.glob("*.yaml"):
    with open(yaml_file, "r", encoding="utf-8") as file:
        try:
            data = yaml.safe_load(file)
            
            # Ensure data is not None and contains AgroManagement
            if not data or "AgroManagement" not in data:
                print(f"Skipped {yaml_file.name}: Missing 'AgroManagement' key or invalid structure.")
                continue
            
            agro_management = data.get("AgroManagement", [])
            
            # Extract variety_name where crop_name is "potato"
            for entry in agro_management:
                for date, details in entry.items():
                    # Skip entries where details is None (empty campaigns)
                    if details is None:
                        continue
                    
                    crop_calendar = details.get("CropCalendar", {})
                    if crop_calendar and crop_calendar.get("crop_name") == "potato":
                        variety_name = crop_calendar.get("variety_name")
                        if variety_name:
                            potato_varieties.append(variety_name)
        except Exception as e:
            print(f"Error reading {yaml_file.name}: {e}")


# Remove duplicates and print the results
unique_varieties = sorted(set(potato_varieties))
print(f"\nFound {len(unique_varieties)} unique potato varieties:")
for variety in unique_varieties:
    print(f"  - {variety}")


Found 5 unique potato varieties:
  - Festien
  - Fontane
  - Innovator
  - Markies
  - Potato_701
